In [ ]:
# ⚙️ Global Config & Services (using centralized modules)

import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    validate_api_keys,
    print_config_summary
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


📂 Working directory: c:\Users\viraj\Zuu\Ledger_mind
✅ Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:298: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:298: UserWarning: ⚠️  OPENROUTER_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:298: UserWarning: ⚠️  GOOGLE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\viraj\Zuu\Ledger_mind\src\services\llm_services.py:298: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


In [ ]:
# Initialize LLM using factory from llm_services
llm = get_llm(config)
print(f"LLM initialized: {config['llm_provider']} / {config['llm_model']}")

# Verify API key with test completion
print("\n🔍 Testing API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"API key verified: {test_msg[:50]}")
except Exception as e:
    print(f" API key test failed: {e}")
    print("Please check your .env file and API key configuration.")


✅ LLM initialized: groq / llama-3.1-8b-instant

🔍 Testing API connection...
✅ API key verified: API working!


In [ ]:
import glob
from pypdf import PdfReader

text_dir = Path(config["data_root"])
text_dir.mkdir(parents=True, exist_ok=True)

doc_files = list(text_dir.glob("*.pdf"))
print(doc_files)

documents = []
for fpath in doc_files:
    reader = PdfReader(str(fpath))
    pdf_text = "\n".join((page.extract_text() or "") for page in reader.pages)

    documents.append({
        "source": fpath.name,
        "content": pdf_text
    })

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc['source']}: {len(doc['content'])} chars")


[WindowsPath('data/annual_report_2024.pdf')]
✅ Loaded 1 documents
  - annual_report_2024.pdf: 640073 chars
